# Modelo de Recomendación de Alcance — 3. Matriz de recomendación

Para cada NIC de `dbanalitica.recomendacion_aire_scr`, la probabilidad de que **su próxima
suspensión** salga efectiva con cada una de las 8 canastas, y la canasta recomendada.

**Requisitos:** notebook 1 corrido (`datos/historico_mo_gestionadas.parquet`) y notebook 2
(`modelos/Modelo_Alcance.cbm`). El histórico se toma del mismo `.parquet` del notebook 1: si se
borró para refrescar, correr primero el 1 y el 2.

## Cómo se arma

Cada NIC se convierte en la fila de "la próxima orden" que el notebook 1 habría construido: todo su
historial cuenta como "previo", y la orden anterior es la última que tuvo. Esa fila se repite
**una vez por canasta**, cambiando solo lo que depende de la canasta (la canasta, el historial del
NIC con ella, la efectividad del barrio, el municipio y el transformador con ella, y el técnico), y
el modelo puntúa las 8. Como al recomendar todavía no se sabe qué técnico va, se usa el **técnico
típico** de cada canasta: su efectividad general y en el municipio.

La **propensión de pago** del servicio externo entra como variable, con una regla forzada en el
entrenamiento: a más propensión de pago, el % solo puede bajar. Así la matriz nunca da un % alto a
un cliente que el servicio dice que va a pagar cuando llegue la brigada.

## Reglas acordadas

- **Se recomienda solo entre 5 canastas.** Pesada Disponible, (D) Brigada Pesada nocturna y Brigada
  Canasta tienen pocas órdenes recientes y el modelo las sobreestima 4 a 12 puntos: su % se muestra,
  pero no pueden ser la recomendada.
- **Se excluyen los que pagaron después de la última suspensión y siguen sin reconexión efectiva**
  ("pagaron antes", según la reunión).

## Cómo leer la matriz

El alcance no se asigna al azar: una probabilidad alta dice *"a suspensiones parecidas les fue mejor
con esta canasta"*, no que mandarla cause el éxito. Y el modelo **subestima la Brigada Pesada unos
5 puntos** en los últimos meses (notebook 2, sección 5): cuando la recomendada le gana a la Pesada
por menos de eso, la diferencia no es concluyente.

## 1. Carga

In [ ]:
import json
import os

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from dotenv import load_dotenv
from sqlalchemy import create_engine

from comun import (ACUMULADOS, SIN_HISTORIA, preparar_historia, puntuar_canastas,
                   variables_canasta_nic, variables_nic)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 250)

DIR_DATOS = 'datos'
DIR_MODELOS = 'modelos'
DIR_SALIDAS = 'salidas'
os.makedirs(DIR_SALIDAS, exist_ok=True)

with open(f'{DIR_MODELOS}/Metadata_Modelo_Alcance.json', encoding='utf-8') as f:
    meta = json.load(f)
FEATURES = meta['features']
CATEGORICAS = meta['categoricas']
CANASTAS = meta['canastas']

modelo = CatBoostClassifier()
modelo.load_model(f'{DIR_MODELOS}/Modelo_Alcance.cbm')

h = preparar_historia(pd.read_parquet(f'{DIR_DATOS}/historico_mo_gestionadas.parquet'))
# La fecha de corte es la del último cierre del histórico, no la de hoy: los "días sin orden" se
# miden contra lo que el modelo conoce.
CORTE = h['fecha_cierre'].max()
print(f'{len(h):,} órdenes gestionadas | {h["nic"].nunique():,} NICs | corte {CORTE.date()}')

## 2. Variables de la próxima orden (`comun.variables_nic`)

Lo que no depende de la canasta. Equivale a la fila nueva del notebook 1: los acumulados "previos"
son el historial completo y la "orden anterior" es la última. Vive en `comun.py` porque también lo
usan las evaluaciones hacia atrás (qué habría recomendado el modelo en un mes pasado).

Dos aproximaciones, porque de la orden futura no se sabe todo:
- **Deuda y facturas** son las de la **última suspensión** del NIC, no las de su última orden: esa
  suele ser una reconexión hecha justo después de pagar, con la deuda casi en cero, y el modelo
  subestimaba ~16 puntos (probado con octubre de 2026). La deuda al despachar todavía no existe.
- **Tipo de orden**: TO504 si el NIC ya tuvo alguna suspensión MI/MS, si no TO501.

## 3. Chequeo: estas variables son las mismas del entrenamiento

Si `variables_nic` construyera algo distinto de lo que el modelo vio en el notebook 1, la matriz
saldría mal sin ningún error a la vista. Se comprueba así: para los NICs cuya última orden es una
suspensión del dataset de entrenamiento, se borra esa orden del histórico, se recalculan sus
variables "como si fuera la próxima" con corte en su fecha, y se comparan con las que el notebook 1
le calculó a esa fila. Tienen que coincidir.

Quedan fuera del chequeo la deuda y el tipo de orden (aquí se aproximan a propósito) y la ubicación
y la tarifa, que en el entrenamiento son las de la propia orden.

In [ ]:
uni = pd.read_parquet(f'{DIR_DATOS}/df_universo_alcance.parquet', columns=['id'] + FEATURES)
ultimas = h.groupby('nic', sort=False).tail(1)
ultimas = ultimas[ultimas['id'].isin(uni['id'])]
muestra = ultimas.sample(min(20000, len(ultimas)), random_state=42)

h_sin = h[h['nic'].isin(muestra['nic']) & ~h['id'].isin(muestra['id'])]
corte_nic = muestra.set_index('nic')['fecha_cierre']
v_chk = variables_nic(h_sin, corte_nic.loc[h_sin['nic'].unique()])
pc_chk = variables_canasta_nic(h_sin)

esperado = uni.set_index('id').loc[muestra['id']]
esperado.index = pd.Index(muestra['nic'].tolist())
obtenido = v_chk.reindex(esperado.index)
claves = list(zip(esperado.index, muestra['CANASTA']))
obtenido['ORDENES_PREVIAS_CANASTA'] = pc_chk['ORDENES_PREVIAS_CANASTA'].reindex(claves).fillna(0).values
obtenido['EF_PREVIAS_CANASTA'] = pc_chk['EF_PREVIAS_CANASTA'].reindex(claves).fillna(0).values

COMPARAR = list(ACUMULADOS.values()) + ['ORDENES_PREVIAS', 'TASA_EF_HIST', 'TASA_EF_SUSP_HIST',
                                        'DIAS_DESDE_ANTERIOR', 'SIN_ORDEN_90D',
                                        'ORDENES_PREVIAS_CANASTA', 'EF_PREVIAS_CANASTA'] + SIN_HISTORIA
# Los NIC sin ninguna orden previa no quedan en h_sin: en el entrenamiento son la fila de su primera orden.
con_historia = esperado['ORDENES_PREVIAS'] > 0
distintas = {}
for c in COMPARAR:
    a, b = esperado.loc[con_historia, c], obtenido.loc[con_historia, c]
    iguales = (a == b) | (a.isna() & b.isna())
    if not iguales.all():
        distintas[c] = int((~iguales).sum())

print(f'{con_historia.sum():,} NICs comparados')
assert not distintas, f'Las variables de predicción no coinciden con las de entrenamiento: {distintas}'
print('Coinciden todas.')

## 4. Puntuación por canasta (`comun.puntuar_canastas`)

Agrega la efectividad de las suspensiones por barrio y municipio, con y sin canasta, suavizada igual
que en el notebook 1, y puntúa las 8 canastas para cada NIC.

### 4.1 Propensión de pago

La del **servicio de propensión de pago** al corte. Se corren sus mismos modelos y su mismo código
(`propension.py`), porque consultarlo NIC por NIC tarda más de 2 horas; una muestra se compara
contra el servicio en producción para comprobar que el número es el mismo.

In [ ]:
from propension import datos_servicio, propension_hoy

load_dotenv('.env')
engine = create_engine(os.environ['SCR_DATABASE_URL'])
nics_h = h['nic'].unique().tolist()
balanza, mo = datos_servicio(engine, nics_h, DIR_DATOS)
propension = propension_hoy(balanza, mo, CORTE, nics_h)
propension.describe().round(3)

In [ ]:
base, P = puntuar_canastas(h, CORTE, modelo, FEATURES, CATEGORICAS, CANASTAS, propension)
print('NICs sin propensión (se les puso la mediana):', int(base['SIN_PROPENSION'].sum()))
P.describe().T[['mean', 'min', 'max']].round(3)

## 5. Exclusión de los que "pagaron antes" y recomendación

La reconexión que cuenta es la **efectiva**: la vista base da por reconectado a quien solo tuvo una
reconexión fallida, y con eso deja fuera de esta exclusión a clientes que pagaron y siguen
suspendidos.

In [ ]:
load_dotenv('.env')
url = os.getenv('SCR_DATABASE_URL')
if not url:
    raise RuntimeError('Falta SCR_DATABASE_URL: copia el .env de "Recomendacion Social" a esta carpeta.')

vista = pd.read_sql(
    'SELECT nic, fecha_ultima_suspension, fecha_pago_post_suspension FROM dbanalitica.recomendacion_aire_scr',
    create_engine(url))
vista['nic'] = vista['nic'].astype(str)
vista = vista.set_index('nic')

reconexion = (h[(h['tipo_os'] == 'TO502') & (h['estado_norm'] == 'EFECTIVA')]
                .groupby('nic')['fecha_cierre'].max())
vista['reconexion_efectiva'] = reconexion
pagaron_antes = (vista['fecha_pago_post_suspension'].notna()
                 & (vista['reconexion_efectiva'].isna()
                    | (vista['reconexion_efectiva'] < pd.to_datetime(vista['fecha_ultima_suspension']))))

nics = vista.index[~pagaron_antes].intersection(P.index)
print(f'NICs en la vista: {len(vista):,} | pagaron antes (fuera): {pagaron_antes.sum():,} | en la matriz: {len(nics):,}')
faltan = vista.index[~pagaron_antes].difference(P.index)
print(f'NICs de la vista sin histórico en el parquet: {len(faltan):,}  (si son muchos, refrescar el notebook 1)')

In [ ]:
# Las tres con pocas órdenes recientes se muestran pero no compiten (regla acordada).
NO_RECOMENDABLES = ['Pesada Disponible', '(D) Brigada Pesada', 'Brigada Canasta']
ELEGIBLES = [c for c in CANASTAS if c not in NO_RECOMENDABLES]

Pn = P.loc[nics]
orden = np.argsort(-Pn[ELEGIBLES].values, axis=1)
top1 = np.array(ELEGIBLES)[orden[:, 0]]
top2 = np.array(ELEGIBLES)[orden[:, 1]]
p1 = np.take_along_axis(Pn[ELEGIBLES].values, orden[:, :1], axis=1)[:, 0]
p2 = np.take_along_axis(Pn[ELEGIBLES].values, orden[:, 1:2], axis=1)[:, 0]

info = base.loc[nics]
matriz = pd.DataFrame({
    'nic': nics,
    'territorio': info['territorio'].values,
    'zona': info['zona'].values,
    'municipio': info['municipio'].values,
    'barrio': info['localidad_barrio'].values,
    'alcance_recomendado': top1,
    'pct_recomendado': (p1 * 100).round(1),
    'segundo_alcance': top2,
    'pct_segundo': (p2 * 100).round(1),
    'ventaja_pp': ((p1 - p2) * 100).round(1),
    'ventaja_sobre_pesada_pp': ((p1 - Pn['Brigada Pesada'].values) * 100).round(1),
})
for c in CANASTAS:
    sufijo = ' (no recomendable)' if c in NO_RECOMENDABLES else ''
    matriz[f'pct {c}{sufijo}'] = (Pn[c].values * 100).round(1)
matriz['prob_pago_con_intervencion'] = (info['PROB_PAGO_CON_INTERVENCION'].values * 100).round(1)
# En SCR el servicio no tiene modelo de "no pago": lo devuelve como 1 - con intervención. Se muestra
# igual que el servicio, pero no entra al modelo (repetiría la otra variable).
matriz['prob_no_pago_con_intervencion'] = 100 - matriz['prob_pago_con_intervencion']
matriz['prob_pago_sin_intervencion'] = (info['PROB_PAGO_SIN_INTERVENCION'].values * 100).round(1)
matriz['sin_propension'] = info['SIN_PROPENSION'].values
matriz['canasta_ultima_orden'] = info['CANASTA_ANTERIOR'].values
matriz['resultado_ultima_orden'] = info['RESULTADO_ANTERIOR'].values
matriz['dias_sin_orden'] = info['DIAS_DESDE_ANTERIOR'].values
matriz['ordenes_previas'] = info['ORDENES_PREVIAS'].values
matriz = matriz.sort_values('pct_recomendado', ascending=False).reset_index(drop=True)
matriz.head()

## 6. ¿La recomendación discrimina?

Lo que hizo fallar a la fórmula fija: recomendaba Brigada Pesada al 99,6 % de los NICs. Aquí se
mira cuánto varía, y qué tan holgada es cada recomendación.

In [ ]:
print('--- Alcance recomendado ---')
print(matriz['alcance_recomendado'].value_counts().to_frame('NICs')
        .assign(pct=lambda d: (d['NICs'] / len(matriz) * 100).round(1)))

print('\n--- Por territorio (% de NICs) ---')
print((pd.crosstab(matriz['territorio'], matriz['alcance_recomendado'], normalize='index') * 100).round(1))

In [ ]:
cerca = matriz['ventaja_pp'] < 2
no_pesada = matriz['alcance_recomendado'] != 'Brigada Pesada'
print(f'Recomendaciones con menos de 2 pp sobre la segunda: {cerca.mean():.1%}  (casi empate)')
print(f'Recomiendan algo distinto de Pesada: {no_pesada.sum():,}; de esas, le ganan a la Pesada por '
      f'menos de 5 pp (el desvío del modelo): {(no_pesada & (matriz["ventaja_sobre_pesada_pp"] < 5)).sum():,}')

ganaria = P.loc[nics, NO_RECOMENDABLES].max(axis=1).values * 100 > matriz.set_index('nic').loc[nics, 'pct_recomendado'].values
print(f'NICs donde una no recomendable habría ganado: {ganaria.sum():,} ({ganaria.mean():.1%})')

print('\n--- Recomendado vs. canasta de la última orden (% de NICs por canasta anterior) ---')
print((pd.crosstab(matriz['canasta_ultima_orden'], matriz['alcance_recomendado'], normalize='index') * 100).round(1))

## 7. Guardado

CSV con BOM para que Excel abra bien las tildes. Va a `salidas/`, que git ignora (`*.csv`).

In [ ]:
archivo = f'{DIR_SALIDAS}/matriz_recomendacion_alcance_{CORTE.date()}.csv'
matriz.to_csv(archivo, index=False, encoding='utf-8-sig')
print(f'guardado: {archivo}  ({len(matriz):,} NICs)')

## 8. Excel de probabilidad de efectividad (sin elegir brigada)

Un solo número por NIC, para **priorizar**, no para decidir con qué brigada ir: la probabilidad de
que su próxima suspensión salga efectiva **si va la Brigada Pesada**, la que más se manda. Se fija
la brigada a propósito: así el número de un NIC se puede comparar con el de otro.

La hoja «Validación octubre» sale de `salidas/matriz_octubre_2026_vs_real.xlsx` (modelo entrenado
hasta septiembre, puntuando las suspensiones de octubre): muestra, por nivel, cuánto predijo el
modelo y cuánto salió efectivo de verdad. Si ese archivo no está, la hoja se omite.

**Pronóstico.** Siempre responde: Efectiva desde 50 %, No efectiva por debajo. Se probó una franja
«Incierta» de 40 a 60 % (acertaba 75 % en el resto, contra 70 % respondiendo todas) y se descartó
porque el negocio necesita respuesta para cada orden. Las que están cerca del 50 % son las de menos
confianza: la probabilidad queda en el archivo para verlo.

**Técnico.** Aquí todavía no se sabe quién va a ir, así que se usa el técnico típico de la Brigada
Pesada. Con la orden ya asignada, `puntuar_canastas(..., tecnicos=...)` usa el técnico real.

In [ ]:
NIVELES = [0, 50, 101]
ETIQUETAS = ['No efectiva (menos de 50 %)', 'Efectiva (50 % o más)']

prob = pd.DataFrame({
    'NIC': nics,
    'Territorio': info['territorio'].values,
    'Zona': info['zona'].values,
    'Municipio': info['municipio'].values,
    'Barrio': info['localidad_barrio'].values,
    'Probabilidad de efectividad %': (Pn['Brigada Pesada'].values * 100).round(1),
    'Propensión de pago con intervención %': (info['PROB_PAGO_CON_INTERVENCION'].values * 100).round(1),
    'No paga aunque le intervengan %': (100 - (info['PROB_PAGO_CON_INTERVENCION'].values * 100).round(1)).round(1),
    'Propensión de pago sin intervención %': (info['PROB_PAGO_SIN_INTERVENCION'].values * 100).round(1),
    'Deuda (última suspensión)': info['deuda_act'].round(0).values,
    'Días sin orden': info['DIAS_DESDE_ANTERIOR'].values,
    'Órdenes previas': info['ORDENES_PREVIAS'].values,
    'Resultado última orden': info['RESULTADO_ANTERIOR'].values,
})
prob['Pronóstico'] = pd.cut(prob['Probabilidad de efectividad %'], NIVELES, right=False, labels=ETIQUETAS).astype(str)
prob = prob.sort_values('Probabilidad de efectividad %', ascending=False).reset_index(drop=True)
prob.insert(0, 'Prioridad', range(1, len(prob) + 1))

resumen_nivel = (prob.groupby('Pronóstico')['Probabilidad de efectividad %']
                   .agg(NICs='size', probabilidad_media='mean')
                   .reindex(ETIQUETAS[::-1]).round(1).reset_index())

ARCHIVO_OCT = f'{DIR_SALIDAS}/matriz_octubre_2026_vs_real.xlsx'
validacion = None
if os.path.exists(ARCHIVO_OCT):
    o = pd.read_excel(ARCHIVO_OCT, sheet_name='Octubre').dropna(subset=['% con la brigada que fue'])
    o['Pronóstico'] = pd.cut(o['% con la brigada que fue'], NIVELES, right=False, labels=ETIQUETAS).astype(str)
    o['efectiva'] = o['Resultado'].eq('Efectiva')
    # Acierta = dijo Efectiva y salió, o dijo No efectiva y no salió.
    o['acierto'] = (o['Pronóstico'] == ETIQUETAS[1]) == o['efectiva']
    validacion = (o.groupby('Pronóstico')
                   .agg(Suspensiones=('efectiva', 'size'),
                        **{'Predijo el modelo %': ('% con la brigada que fue', 'mean'),
                           'Salió efectiva de verdad %': ('efectiva', lambda s: s.mean() * 100),
                           'Acertó %': ('acierto', lambda s: s.mean() * 100)})
                   .reindex(ETIQUETAS[::-1]).round(1).reset_index())
    print(f'Octubre: acertó {o["acierto"].mean():.1%}')

print(resumen_nivel.to_string(index=False))
print()
print(validacion.to_string(index=False) if validacion is not None else 'sin archivo de octubre')

In [ ]:
notas = pd.DataFrame({'Cómo leer este archivo': [
    f'Probabilidad de que la próxima suspensión de cada NIC salga efectiva, calculada con datos al {CORTE.date()}.',
    'Supone que va la Brigada Pesada, la que más se manda, con un técnico típico. Sirve para priorizar clientes, NO para elegir la brigada.',
    'Pronóstico: Efectiva (50 % o más) o No efectiva (menos de 50 %). Cuanto más cerca del 50 %, menos seguro está el modelo.',
    'Prioridad 1 = el NIC con más probabilidad de que la suspensión salga efectiva.',
    'La propensión de pago es la del servicio de propensión. El modelo está obligado a que nunca se contradigan: a más propensión de pago, menos probabilidad de efectividad.',
    'Eso no quiere decir que sean opuestos exactos: dos clientes con la misma propensión pueden tener % distinto por su historial, el barrio o el transformador.',
    'Es una probabilidad, no una garantía: con 70 %, de cada 100 casos parecidos unos 30 salen mal igual.',
    'El modelo viene subestimando a la Brigada Pesada unos 5 puntos: la efectividad real tiende a ser algo mayor.',
    'No incluye a los que pagaron después de la última suspensión y siguen sin reconexión efectiva.',
    'Se recalcula cada mes: después de un mes el número ya va desfasado.',
]})

archivo_xlsx = f'{DIR_SALIDAS}/probabilidad_efectividad_{CORTE.date()}.xlsx'
with pd.ExcelWriter(archivo_xlsx, engine='openpyxl') as xw:
    fila = 0
    resumen_nivel.to_excel(xw, sheet_name='Resumen', index=False, startrow=fila)
    fila += len(resumen_nivel) + 3
    if validacion is not None:
        pd.DataFrame({'Validación con octubre 2026 (modelo entrenado hasta septiembre)': []}).to_excel(
            xw, sheet_name='Resumen', index=False, startrow=fila)
        validacion.to_excel(xw, sheet_name='Resumen', index=False, startrow=fila + 1)
        fila += len(validacion) + 4
    notas.to_excel(xw, sheet_name='Resumen', index=False, startrow=fila)
    prob.to_excel(xw, sheet_name='Probabilidad por NIC', index=False)
    for hoja in xw.sheets.values():
        hoja.freeze_panes = 'A2'
        for col in hoja.columns:
            ancho = max(len(str(c.value)) if c.value is not None else 0 for c in col[:300])
            hoja.column_dimensions[col[0].column_letter].width = min(ancho + 2, 70)
print(f'guardado: {archivo_xlsx}  ({len(prob):,} NICs)')